In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")


✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_pythWL,dog_pythWL,fav_Luck,dog_Luck,fav_last10,dog_last10,fav_last20,dog_last20,fav_last30,dog_last30,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-08T13:40:00-04:00,DET,None,CHC,None,DET,None,CHC,None,-1.5,1.5,0.364,0.654,175,-189,1,DET -1.5,Jack Flaherty,Cade Horton,3.72,3.51,1.09,1.21,76,19,65.1,25.2,0.212,0.263,0.636,0.625,.250,.261,3.20,3.66,42,40,24,24,L 1,W 1,4.9,5.7,3.5,4.0,0.0,-0.1,1.3,1.6,42-24,42-22,0.0,-2.0,6-4,7-3,12-8,15-5,19-11,19-11,4.86,5.67,2499,2506,2234,2227,321,363,558,581,100,120,11,15,63,70,307,355,24,82,216,234,587,502,.250,.261,.321,.333,.406,.447,.728,.780,106,124,3.53,3.97,3.20,3.66,21,15,498,520,233,254,209,233,186,182,556,498,125,104,3.66,3.98,1.165,1.226,7.6,8.2
1,2025-06-08T13:35:00-04:00,TEX,None,WAS,None,WAS,None,TEX,None,-1.5,1.5,0.426,0.583,135,-140,0,WAS +1.5,Jacob Latz,Trevor Williams,2.95,6.03,1.42,1.44,20,47,18.1,59.2,0.254,0.293,0.462,0.469,.220,.239,3.12,4.92,30,30,35,34,W 1,L 1,3.3,4.3,3.3,5.0,0.0,-0.1,0.0,-0.9,33-32,27-37,-3.0,3.0,4-6,6-4,6-14,12-8,13-17,15-15,3.34,4.27,2303,2354,2096,2118,217,273,461,507,74,106,6,10,55,61,211,260,57,57,166,178,536,485,.220,.239,.282,.306,.353,.387,.634,.693,83,99,3.34,5.03,3.12,4.92,17,19,459,557,217,322,196,308,180,219,505,508,121,81,3.60,4.06,1.131,1.378,7.3,8.9
2,2025-06-08T13:35:00-04:00,PIT,None,PHI,None,PIT,None,PHI,None,-1.5,1.5,0.345,0.677,190,-210,1,PIT -1.5,Paul Skenes,Cristopher Sánchez,2.05,3.15,0.88,1.31,85,74,83.1,65.2,0.181,0.240,0.385,0.578,.227,.252,3.90,4.04,25,37,40,27,W 2,L 4,3.2,4.6,4.1,4.4,0.3,-0.3,-0.7,0.0,25-40,34-30,0.0,3.0,6-4,2-8,10-10,11-9,13-17,18-12,3.17,4.62,2413,2451,2148,2181,206,296,487,549,80,98,12,9,53,66,199,281,61,58,222,228,569,507,.227,.252,.305,.327,.339,.398,.644,.725,81,102,4.12,4.38,3.90,4.04,14,20,506,555,268,280,250,257,197,196,481,599,107,102,3.75,3.60,1.218,1.311,7.9,8.7
3,2025-06-08T13:05:00-04:00,MIL,None,SD,None,MIL,None,SD,None,-1.5,1.5,0.444,0.573,125,-134,1,MIL -1.5,Freddy Peralta,Ryan Bergert,2.92,2.00,1.21,1.00,75,4,71.0,9.0,0.216,0.219,0.538,0.571,.238,.248,3.86,3.48,35,36,30,27,W 1,L 1,4.5,4.1,4.1,3.8,0.0,-0.1,0.4,0.2,35-30,33-30,0.0,3.0,8-2,5-5,14-6,9-11,18-12,14-16,4.49,4.10,2458,2342,2171,2099,292,258,517,521,89,90,5,8,67,63,269,238,84,49,227,194,522,449,.238,.248,.316,.314,.365,.377,.681,.691,92,93,4.12,3.83,3.86,3.48,15,23,520,467,268,241,247,216,237,207,534,544,104,116,4.07,3.83,1.316,1.206,8.1,7.5
4,2025-06-08T13:40:00-04:00,CLE,None,HOU,None,CLE,None,HOU,None,-1.5,1.5,0.387,0.638,158,-176,1,CLE -1.5,Tanner Bibee,Brandon Walter,3.89,0.00,1.24,0.80,56,5,69.1,5.0,0.241,0.176,0.524,0.562,.233,.251,3.98,3.52,33,36,30,28,L 3,W 3,3.9,4.0,4.2,3.7,0.1,0.1,-0.2,0.5,29-34,35-29,4.0,1.0,4-6,7-3,8-12,13-7,14-16,19-11,3.90,4.05,2315,2368,2058,2130,246,259,480,535,87,86,6,6,69,61,233,247,47,35,202,185,516,487,.233,.251,.306,.317,.376,.389,.682,.706,93,99,4.24,3.66,3.98,3.52,18,19,530,455,267,234,246,221,234,201,534,616,103,116,4.12,3.48,1.375,1.162,8.6,7.3
5,2025-06-08T15:10:00-04:00,NYM,None,COL,None,COL,None,NYM,None,-1.5,1.5,0.636,0.396,-175,153,0,COL +1.5

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-08.csv
